# EDA de datos del TFM

Este notebook sustituye al EDA exploratorio interactivo para la parte formal del TFM. Trabaja solo con los CSV versionados en `data/raw`, por lo que no depende de descargar datos en vivo.


## 1. Objetivo

Comprobar la calidad, cobertura y utilidad de los datos financieros usados por el MVP: estructura de los CSV, nulos, duplicados, rango temporal, variedad de activos, retornos, volatilidad y drawdown.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

case_summary = pd.read_csv(PROCESSED_DIR / "eda_resumen_ficheros.csv")
series_summary = pd.read_csv(PROCESSED_DIR / "eda_resumen_series.csv")
long_df = pd.read_csv(PROCESSED_DIR / "eda_dataset_largo.csv", parse_dates=["datetime"])

case_summary


## 2. Normalización aplicada

Los CSV de `yfinance` tienen columnas multinivel: ticker y campo OHLCV. El generador `scripts/generate_data_eda.py` los lee con `header=[0, 1]`, conserva los metadatos y genera un dataset largo en `data/processed/eda_dataset_largo.csv`.


In [ ]:
long_df.head()


## 3. Cobertura y calidad

La tabla siguiente resume filas, rango observado, porcentaje de nulos y duplicados por fecha para cada CSV.


In [ ]:
case_summary[[
    "source_file", "analysis_goal", "tickers", "interval", "rows",
    "start_observed", "end_observed", "missing_pct", "duplicated_dates"
]]


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
case_summary.sort_values("rows").plot.barh(x="source_file", y="rows", ax=ax, legend=False)
ax.set_title("Observaciones por fichero")
ax.set_xlabel("Filas")
ax.set_ylabel("")
plt.tight_layout()


## 4. Resumen financiero por serie

Se calculan rentabilidad total, volatilidad anualizada aproximada y drawdown máximo por cada combinación fichero-ticker.


In [ ]:
series_summary[[
    "source_file", "ticker", "observations", "start_observed", "end_observed",
    "total_return_pct", "volatility_ann_pct", "max_drawdown_pct"
]].sort_values(["source_file", "ticker"])


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_df = series_summary.sort_values("total_return_pct")
ax.barh(plot_df["ticker"] + " | " + plot_df["source_file"].str.slice(0, 24), plot_df["total_return_pct"])
ax.set_title("Rentabilidad total por serie")
ax.set_xlabel("Rentabilidad total (%)")
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_df = series_summary.sort_values("max_drawdown_pct")
ax.barh(plot_df["ticker"] + " | " + plot_df["source_file"].str.slice(0, 24), plot_df["max_drawdown_pct"])
ax.set_title("Drawdown maximo por serie")
ax.set_xlabel("Drawdown maximo (%)")
plt.tight_layout()


## 5. Ejemplos de series de cierre

Este bloque permite inspeccionar visualmente varias series normalizadas a base 100. Se agrupa por caso para evitar mezclar horizontes temporales incompatibles.


In [ ]:
for case_id, group in long_df.groupby("case_id"):
    pivot = group.pivot(index="datetime", columns="ticker", values="Close").dropna(how="all")
    if pivot.empty:
        continue
    norm = pivot / pivot.iloc[0] * 100
    ax = norm.plot(figsize=(9, 4), title=f"Evolucion normalizada base 100 - {case_id}")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Base 100")
    plt.tight_layout()
    plt.show()


## 6. Conclusiones para la memoria

- El dataset cubre acciones, ETFs, indice amplio, criptoactivo, divisa y futuro de materia prima.
- La capa de normalizacion es necesaria porque la salida de `yfinance` no es plana.
- La calidad de `Close` es suficiente para las metricas principales del MVP.
- El volumen debe usarse con prudencia en divisas e instrumentos intradia.
- Para versiones finales conviene fijar rangos absolutos cuando sea posible, ya que los periodos relativos dependen de la fecha de descarga.
